In [ ]:
!git clone https://github.com/aria-yang/youtube-thumbnail-performance-predictor.git
%cd /content/youtube-thumbnail-performance-predictor


Install Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q numpy pandas pillow tqdm scikit-learn loguru typer python-dotenv
!pip install -q easyocr facenet-pytorch deepface

Run full merged pipeline

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os, shutil
from pathlib import Path

# Define the source directory on Google Drive
artifact_root = Path(
    "/content/drive/MyDrive/youtube-thumbnail-performance-predictor-artifacts"
)

# Define the destination directory in the cloned repository
local_processed_data_path = Path(
    "/content/youtube-thumbnail-performance-predictor/data/processed"
)
local_processed_data_path.mkdir(parents=True, exist_ok=True)

# List of files to copy from Google Drive to local processed data folder
files_to_copy_from_drive = [
    "merged_cnn_cache_resnet50.csv",
    "merged_cnn_embeddings_resnet50.npy",
    "merged_labeled_data.csv",
    "merged_ocr_features.csv",
    "merged_text_embeddings.npy",
    "merged_face_cache.csv",
    "merged_face_embeddings.npy",
]

for filename in files_to_copy_from_drive:
    src_path = artifact_root / filename
    dst_path = local_processed_data_path / filename
    if src_path.exists():
        shutil.copy2(src_path, dst_path)
        print(f"Copied {filename} from Google Drive to local data/processed")
    else:
        print(f"Missing in Google Drive artifacts: {filename}")

In [ ]:
!python training/train_fusion.py \
  --device cuda \
  --split_name random \
  --num_epochs 40 \
  --lr 1e-3 \
  --early_stopping_metric auroc \
  --early_stopping_patience 12 \
  --seed 42

## Tuning

In [ ]:
!python training/finetune_fusion_end_to_end.py \
  --device cuda \
  --split_name random \
  --batch_size 32 \
  --num_epochs 20 \
  --freeze_epochs 5 \
  --unfreeze_all_epoch 5 \
  --head_lr 5e-4 \
  --backbone_lr 1e-5 \
  --weight_decay 1e-4 \
  --hidden1 512 \
  --hidden2 256 \
  --dropout_p 0.35 \
  --checkpoint_path models/fusion_end_to_end_resnet50_head_only_proj_lr5e4.pt \
  --history_path data/processed/fusion_end_to_end_head_only_proj_lr5e4_history.csv

# Ablation and SHAP

In [ ]:
!python -m training.ablation_study

In [ ]:
!rm -f models/fusion_mlp_shap.pt
!python -m training.run_shap_analysis

# Cross-Validation

In [ ]:
!python -m training.eval_crosssplit

In [ ]:
import os, shutil
from pathlib import Path

artifact_root = Path(
    "/content/drive/MyDrive/youtube-thumbnail-performance-predictor-artifacts"
)
artifact_root.mkdir(parents=True, exist_ok=True)

local_processed_data_path = Path(
    "/content/youtube-thumbnail-performance-predictor/data/processed"
)

# Find all .csv and .npy files in the local processed data directory
files_to_copy = []
for file_path in local_processed_data_path.glob("*.csv"):
    files_to_copy.append(file_path)
for file_path in local_processed_data_path.glob("*.npy"):
    files_to_copy.append(file_path)

for src_path in files_to_copy:
    if src_path.exists():
        shutil.copy2(src_path, artifact_root / src_path.name)
        print(f"Copied {src_path.name}")
    else:
        print(f"Missing locally: {src_path}")